# Exercise 3.1 — Supervised learning

*HDS MSc · Module 3 · the Python version of this exercise; the web page is at <https://inflamuob.github.io/HDS_M3/day3/day3-01-supervised-learning.html>.*

*Open it in Jupyter, VS Code or Positron and run it a cell at a time. Keep it in the folder that holds your `data` folder. Cells with `______` are blanks for you to fill in.*

A supervised learning model makes predictions after seeing many examples that
come with the correct answer, and discovering the connections in the data that
produce those answers. It is like a student preparing for an exam by studying
past papers that contain both the questions and the answers: after enough past
papers, the student is ready to sit a new one. These systems are "supervised"
in the sense that a human gives the system data with the known correct results.

This exercise repeats the pipeline from Day 1, this time stopping after each
coding step to understand what is happening.

### Packages and data

Packages needed for this session:

In [ ]:
# Packages for this exercise. If one is missing, install it once from a terminal
# (in a Jupyter notebook, type %pip instead of pip):
# pip install pandas numpy matplotlib seaborn scikit-learn

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor, plot_tree, export_text  # a new learning algorithm

Load the dataset. `data/glucose_study.csv` is the dataset for the whole of
Day 3. It comes from `data/diabetes.csv`, the file you met on Day 2: we kept two
columns only, age in years and one blood glucose measurement in mg/dL, removed
the five impossible glucose readings of 0, and cut what was left down to the
first 80 rows, which is small enough to see every point on a plot.

In [ ]:
glu = pd.read_csv("data/glucose_study.csv")

glu.head()

Now visualise the relationship:

In [ ]:
fig, ax = plt.subplots()
ax.scatter(glu["age"], glu["glucose"])
ax.set_xlabel("Age (feature)")
ax.set_ylabel("Glucose (label)")
plt.show()

### A supervised learning algorithm: linear regression

Suppose we want to predict a person's glucose level from their age with this
dataset. We can fit a **learning algorithm** to the dataset and create a
**predictive model**. Here the learning algorithm is linear regression. seaborn
can already fit and draw it, through `regplot()`, as you did on Day 1:

In [ ]:
fig, ax = plt.subplots()
sns.regplot(data=glu, x="age", y="glucose", ci=None, ax=ax)
ax.set_xlabel("Age (feature)")
ax.set_ylabel("Glucose (label)")
plt.show()

This amounts to using the `lm()` function to fit the data, as you saw in the
Statistics module
([statistics recap](https://anasrana.github.io/ems-practicals/linear-regression.html)).

To recap, the linear regression algorithm is as follows. In ML we write the
equation as

$$y = \beta_0 + \beta_1 x_1$$

where

- $y$ is the outcome **label**: the column to predict (glucose);
- $\beta_0$ is the intercept of the line, sometimes known as the **bias**. It is
  a **parameter** of the model;
- $\beta_1$ is the **weight** of the feature $x_1$, the same as the slope in the
  equation of a line. It is a **parameter** of the model;
- $x_1$ is the **feature**: the input column (age).

And how do we find the two parameters? They are the result of fitting the
learning algorithm (linear regression) to the data. Once they are fixed we have
the predictive model: if a new sample $x_i$ arrives with its age, we can work
out its predicted glucose $\hat y_i$,

$$\hat y_i = \hat\beta_0 + \hat\beta_1 x_i$$

where the hats mark values that came out of the fit.

As you will see later today, fitting works by finding the $\beta_0$ and
$\beta_1$ that make the **best fit line**. That is calculated by optimising a
loss (cost) function, through either ordinary least squares (OLS) or gradient
descent. Do not worry about this just now!

More specifically, when you use `LinearRegression()` below, or the other standard functions
that fit a linear model in R or Python, $\beta_0$ and $\beta_1$ (up to
$\beta_n$, depending on how many features you include in your model) are
calculated using OLS.

In scikit-learn the standard steps are:

1. Separate the features `X` and the label `y`.
2. Create a model object.
3. Fit the model to the data using `.fit()`.
4. Use `.predict()` to obtain predictions.

In [ ]:
# 1. Separate X (the feature) and y (the label)
X = glu[["age"]]      # two-dimensional: a data frame with one column
y = glu["glucose"]    # one-dimensional: a single column

# 2. Define the model
lin_reg = LinearRegression()

# 3. Fit the model
lin_reg = lin_reg.fit(X, y)

Let's look at the learned parameters:

In [ ]:
intercept = lin_reg.intercept_
slope = lin_reg.coef_[0]

print(intercept, slope)

If we take these coefficients and draw the line ourselves, we obtain the same
thing as the `regplot()` line above.

In [ ]:
ages = np.array([glu["age"].min(), glu["age"].max()])   # the two ends of the line

fig, ax = plt.subplots()
ax.scatter(glu["age"], glu["glucose"])
ax.plot(ages, intercept + slope * ages, color="red", linestyle="--", linewidth=2)
ax.set_xlabel("Age (feature)")
ax.set_ylabel("Glucose (label)")
plt.show()

The predictive model, with its fitted parameters, is then

$$\widehat{\text{glucose}} = 110.30 + 0.289 \times \text{age}$$

If a new sample now arrives with $\text{age} = 45$, what is the predicted
glucose? Can you include it in the plot?

In [ ]:
age_newsample = 45
predicted_glucose = 110.30 + 0.289 * age_newsample
print(round(predicted_glucose, 3))

This is the same as doing:

In [ ]:
# predict() wants a data frame, because it normally receives more than one
# sample and more than one feature. Here it has one row and one column.
lin_reg.predict(pd.DataFrame({"age": [45]}))

The value is 123.29, a touch below our 123.305, because we typed the two
parameters rounded.

In [ ]:
fig, ax = plt.subplots()
ax.scatter(glu["age"], glu["glucose"])
ax.plot(ages, intercept + slope * ages, color="red", linestyle="--", linewidth=2)
ax.scatter(age_newsample, predicted_glucose, color="grey", s=150, zorder=3)
ax.set_xlabel("Age (feature)")
ax.set_ylabel("Glucose (label)")
plt.show()

**Question!** If my new sample has $\text{age} = 10$, can I still predict
glucose, even though nobody in the data used to create the model was 10 years
old?

In [ ]:
age_newsample = 10
predicted_glucose = 110.30 + 0.289 * age_newsample
print(round(predicted_glucose, 3))

In [ ]:
ages = np.array([0, 100])   # draw the line well beyond the data

fig, ax = plt.subplots()
ax.scatter(glu["age"], glu["glucose"])
ax.plot(ages, intercept + slope * ages, color="red", linestyle="--", linewidth=2)
ax.scatter(age_newsample, predicted_glucose, color="grey", s=150, zorder=3)
ax.set_xlim(0, 100)
ax.set_xlabel("Age (feature)")
ax.set_ylabel("Glucose (label)")
plt.show()

Yes we can! How confident we are about the predicted value is another matter,
which we will discuss later on. The model is an infinite line that maps age to
glucose. The youngest person in the data is 21, yet we could even give the model
a negative age, or a very high one (say 1000), and obtain a predicted glucose.
It would make no sense.

### Exercise: understand the role of the dataset in predictive modelling

Let's fit a different learning algorithm, the **decision tree**, to different
numbers of training points. You will learn how decision trees work later in the
module. As you have seen in the previous classes, learning algorithms are
already coded for you as functions, so you can apply one already.

The scikit-learn function is the
[decision tree regressor](https://scikit-learn.org/stable/modules/tree.html).

In [ ]:
# at most 3 questions deep, and at least 5 people in every final group
tree_reg = DecisionTreeRegressor(max_depth=3, min_samples_leaf=5)

tree_reg = tree_reg.fit(X, y)

Check what the fitted model consists of, and print the information the model
encodes. Compare it with the two numbers that were the whole of `lin_reg`.

In [ ]:
print(export_text(tree_reg, feature_names=["age"]))

Now plot the model to get a better idea of what it is doing. This is the
equivalent of printing the intercept and the coefficient of a linear
regression.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
plot_tree(tree_reg, feature_names=["age"], filled=True, rounded=True,
          fontsize=8, ax=ax)
plt.tight_layout()
plt.show()

Get the predicted glucose values:

In [ ]:
glu["predicted_glucose"] = tree_reg.predict(X)

glu.head()

Here we are giving the model the same dataset we fitted the learning algorithm
on, as if its rows were new samples whose glucose we want predicted. Do you see
how the real value and the predicted one differ? The closer we can get them to
each other, the better the model we will have!

> **Stop here**
>
> Make sure you have understood this concept before you go on.

Do these plots help?

In [ ]:
ordered = glu.sort_values("age")   # a step line has to be drawn from left to right

fig, ax = plt.subplots()
ax.scatter(glu["age"], glu["glucose"], s=60, color="blue", alpha=0.7)
ax.step(ordered["age"], ordered["predicted_glucose"], where="post",
        color="darkred", linewidth=2)
ax.set_xlabel("Age (years)")
ax.set_ylabel("Glucose (mg/dL)")
plt.show()

In [ ]:
fig, ax = plt.subplots()
ax.scatter(glu["age"], glu["glucose"], s=60, color="blue", alpha=0.7)
ax.step(ordered["age"], ordered["predicted_glucose"], where="post",
        color="darkred", linewidth=2)
ax.scatter(glu["age"], glu["predicted_glucose"], s=30, color="grey", alpha=0.8, zorder=3)
ax.set_xlabel("Age (years)")
ax.set_ylabel("Glucose (mg/dL)")
plt.show()

> **Discuss**
>
> Discuss this with your partner and explain it to each other.

Now let's fit the decision tree algorithm on an even smaller set of points:

In [ ]:
glu30 = glu.iloc[:30]
glu30.shape

1. What is the model created?
2. How does it differ from the previous one?
3. Do the same with the linear regression model. What changes?

> **Stuck?**
>
> Nothing new is needed. `DecisionTreeRegressor` and `LinearRegression` are the
> learning algorithms, and they have not changed. Fit each of them again, and
> change only the `X` and `y` you fit them on.

> **Learning points**
>
> - A learning algorithm fitted to a dataset gives a predictive model. For linear
>   regression the model is two numbers: the intercept and the slope.
> - A model returns a prediction for any input, including ages far outside the
>   data. Whether that prediction makes sense is for you to judge.
> - The predictive model depends on the learning algorithm you chose and on the
>   dataset it was fitted to.

## Stretch

> **Stretch — a different feature**
>
> Go back to [`data/diabetes.csv`](data/diabetes.csv) and repeat this page with `mass` (body mass
> index) as the feature instead of age, on the same 80 people: drop the rows with
> a glucose of 0 and keep the first 80. What are the two parameters now? Does the
> line follow the points any better than it did for age? Look at the smallest
> values of `mass` before you trust the answer.